# 🏁 EV Grand Prix v11 · Ridge Lap · S6E9

**CPU only · no training · about 4 minutes · plain text and ASCII**

Version 8 strips the Grand Prix down to its fastest part: **blending files whose exact public score is known**.
No engines are trained or stacked here. Every file on the grid is a public notebook output, pinned by its hash to
the version that earned its score. One calibrated Fisher model decides the mix, and the generator's four hard rules
finish it.

| version | idea it added | public LB |
|---|---|---:|
| v1 | 48 OOF engines, nested hill climb, uncalibrated Fisher merge | 0.94655 |
| v2 | 67 engines, 4 generator rules | 0.94657 |
| v3 | pit-stop model: Fisher correlation calibrated on 66 labeled pairs | 0.94657 |
| v4–v5 | timing tower over files with exact public scores | 0.94659 |
| v6–v7 | jazivxt's blend in the tower, ASCII design, version correction | 0.94661 |
| v8 | **blend only**: 20 scored files, Paul Bryan Elefante's logistic model on the grid | 0.94663 |
| v9 | fresh versions of three notebooks (jazivxt V37, nina rank+ML V7, mikhailnaumov V5), re-pinned | 0.94663 |
| v10 | nina2025's Round+Robin blend on the grid, its two tricks explained and tested | 0.94663 |
| **v11** | Paul Bryan Elefante's **generator-aware ridge logistic regression** on the grid, plus an honest OOF hedge (Lap 6) | shown on this page |

> **Honesty note.** This notebook optimizes the **public** leaderboard with public scores that files already have on
> the Code tab. Public scores are exact for the public 20% of the test rows and say less about the private 80%. For a
> final selection, pair this file with one built only from out-of-fold predictions (version 7 of this notebook
> writes one: `submission_oof_only_hill.csv`).

In [ ]:
FLAG_ROW = "|" + "#_" * 38 + "|"
BANNER = "\n".join([FLAG_ROW, "|" + "_#" * 38 + "|", r"""
    _______    __   __________  ___    _   ______     ____  ____  _____  __
   / ____/ |  / /  / ____/ __ \/   |  / | / / __ \   / __ \/ __ \/  _/ |/ /
  / __/  | | / /  / / __/ /_/ / /| | /  |/ / / / /  / /_/ / /_/ // / |   /
 / /___  | |/ /  / /_/ / _, _/ ___ |/ /|  / /_/ /  / ____/ _, _// / /   |
/_____/  |___/   \____/_/ |_/_/  |_/_/ |_/_____/  /_/   /_/ |_/___//_/|_|

      ___ _    ___ _  _ ___     ___  _  _ _ __   __  __   ___ _
     | _ ) |  | __| \| |   \   / _ \| \| | |\ \ / /  \ \ / / / |
     | _ \ |__| _|| .` | |) | | (_) | .` | |_\ V /    \ V /| | |
     |___/____|___|_|\_|___/   \___/|_|\_|____|_|      \_/ |_|_|

      H E L L O ,   K A G G L E R S !
      Ridge lap: a logistic regression that sees the rows differently, and an honest hedge.

     _/=\_          _/=\_          _/=\_          _/=\_          _/=\_
  ° 'o---o'      ° 'o---o'      ° 'o---o'      ° 'o---o'      ° 'o---o'   >>>
 ============================================================================""",
                   FLAG_ROW, "|" + "_#" * 38 + "|",
                   "   20 scored files | hash-pinned | 1 calibrated Fisher model | 4 road rules | 1 podium"])
print(BANNER)

In [ ]:
import time

T0 = time.time()
STAGES = 7


def pit(stage, label):
    """A compact car with an exhaust trail drives stage/STAGES of the way to the flag."""
    width = 60
    pos = int((width - 7) * stage / STAGES)
    trail = "° ° °"[-min(pos, 5):] if pos else ""
    print(f"\n LAP {stage:2d}/{STAGES} · {label} · {time.time() - T0:6.1f}s")
    print(" " + " " * pos + " _/=\\_ " + " " * (width - pos - 7) + " |#_#|")
    print(" " + "=" * (pos - len(trail)) + trail + "'o---o'" + "-" * (width - pos - 7) + " |_#_|")


def box(title, lines, width=82):
    """Draw text inside an ASCII box."""
    print("+" + "-" * (width - 2) + "+")
    print("| " + title.upper().ljust(width - 4) + " |")
    print("+" + "-" * (width - 2) + "+")
    for line in lines:
        print("| " + line.ljust(width - 4) + " |")
    print("+" + "-" * (width - 2) + "+")


def podium(first, second, third):
    """Three finishers, each a (label, detail) tuple."""
    print("\n".join([
        r"                          \o/",
        r"                        .-----.",
        r"            \o/         |  1  |",
        r"          .-----.       |     |         \o/",
        r"          |  2  |       |     |       .-----.",
        r"          |     |       |     |       |  3  |",
        r"   =========================================================",
    ]))
    for place, (label, detail) in zip((1, 2, 3), (first, second, third)):
        print(f"      P{place}  {label:31s} {detail}")


TROPHY = r"""
                 ___________
                '._==_==_=_.'        CHECKERED FLAG!  BLEND-ONLY SPRINT.
                .-\:      /-.        |#_#_#_#_#_#_#_#|
               | (|: V11  |) |       |_#_#_#_#_#_#_#_|
                '-|:      |-'        |#_#_#_#_#_#_#_#|
                  \::.    /          |
                   '::. .'           |        _/=\_
                     ) (             |   ° ° 'o---o'  >>>
                   _.' '._           |
                  '-------'     =====|=======================
"""

pit(0, "grid position")

## Lap 1 · Inputs

The competition data (for ids and the four rule columns), jazivxt's
[`s6e9-zoom-zoom-baseline`](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline) dataset for the public
anchor, and nineteen public notebooks as notebook inputs. Every one of them is listed with its author in Lap 2.

In [ ]:
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm, rankdata

warnings.filterwarnings("ignore")
ID, TARGET = "id", "Will_Buy_EV"
ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)
SEED = 2026
PRIOR_POS_RATE = 0.174645      # train positive rate
K_TOP = 0.2848                 # pit-stop model constant, calibrated on 66 labeled OOF pairs (versions 3-7)


def find(pattern, hint=None):
    """Find one input file. With a hint the path MUST contain it: never fall back silently.
    A hint starting with '=' must equal one whole folder name (so 'h-blend' never matches 'h-blend-rank-ml')."""
    hits = sorted({p for p in ROOT.rglob(pattern) if p.is_file()})
    if hint and hint.startswith("="):
        hits = [p for p in hits if hint[1:] in p.parts]
    elif hint:
        hits = [p for p in hits if hint in str(p).lower()]
    if not hits:
        raise FileNotFoundError(f"Attach the input that contains {pattern} (path hint: {hint})")
    return hits[0]


def sha12(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12]


RULE_COLS = ["Annual_Income_USD", "Daily_Commute_km", "Subsidy_Available",
             "Environmental_Concern_Level", "Range_Anxiety_Level"]
sample = pd.read_csv(find("sample_submission.csv", "playground-series-s6e9"), usecols=[ID])
test = pd.read_csv(find("test.csv", "playground-series-s6e9"), usecols=[ID] + RULE_COLS)
test = test.set_index(ID).loc[sample[ID]].reset_index()
test_ids = sample[ID].to_numpy()
print(f"test rows {len(test_ids):,}")
pit(1, "inputs found")

## Lap 2 · The grid: every file with an exact public score

A public score belongs to **one version** of a notebook, but an attached notebook mounts its **latest** output. So
only notebooks whose latest version is the scored one are on the grid, and each file is pinned by the hash of the
bytes that were checked. If an author changes a notebook later, the file drops out instead of entering the race
with a score that belongs to other bytes.

This is not hypothetical. While version 8 was running, two authors published new versions, and the pins dropped their
changed files on the spot. Since then nina2025's h-blend (0.94661) has gone private, so it is off the grid, and her
rank+ML notebook was renamed. Her **Round+Robin** notebook (0.94663) is on the grid (Lap 5 explains it).

**New in v11: a logistic regression that scores 0.94656.** Paul Bryan Elefante's
[generator-aware ridge logistic regression](https://www.kaggle.com/code/heuljax/kps6e09-generator-aware-ridge-logistic-regression)
is a *linear* model built on features that mirror how the data was generated. It scores almost as high as the blended
files at the top of the Code tab, yet it ranks the contested rows more differently from them than anything we have
seen (1 − effective ρ with the anchor ≈ 0.0022, about 3× the next most different strong file). That is exactly what the
timing tower is looking for.

One special case:

* **A probe in disguise.** One 0.94656 "teacher–student" output is byte-identical to megayak's band-swapped file,
  which was built by *reading* the public split. It is dropped.

🎓 **Pit-stop lesson: titles describe intentions; hashes describe bytes.** Check both the version and the bytes of
anything you did not make yourself.

In [ ]:
# label, file name, path hint, public score of exactly these bytes, pinned sha256 (first 12 hex)
GRID = [
    ("jazivxt anchor", "submission_latest_best.csv", "zoom-zoom-baseline", 0.94657, "7d46cc5b5dc1"),
    ("nina2025 Round+Robin", "submission.csv", "round-robin", 0.94663, "5181a8e5d5f4"),
    ("shawncsx Ens5 fusion F", "submission.csv", "ens5-fusion-f", 0.94661, "d404f854d857"),
    ("kozykappa electric car", "submission_improved.csv", "electric-car-lb", 0.94657, "c487cd585ba5"),
    ("heuljax generator-aware ridge LR", "submission.csv", "generator-aware-ridge", 0.94656, "2d2c104c722f"),
    ("nina2025 rank+ML", "submission.csv", "rank-ml", 0.94660, "7cd77c5529a8"),
    ("jazivxt blend (V37)", "blend.csv", "single-model-zoom-zoom", 0.94660, "280282ed2e5d"),
    ("tamerlan HiRGE blend", "submission.csv", "custom-hirge-net", 0.94649, "a5ae1e649080"),
    ("shawncsx Ens5 fusion E", "submission.csv", "ens5-fusion-e", 0.94647, "14cad83d1a83"),
    ("BlamerX XGB", "submission.csv", "window-encodings", 0.94646, "4fed48ef8217"),
    ("denpugovkin 15 teachers", "submission.csv", "fifteen-teachers", 0.94644, "3f472a5c2796"),
    ("mikhailnaumov XGB (V5)", "submission.csv", "purchases-single-xgb", 0.94643, "45b37ce692b4"),
    ("heuljax XGB", "submission.csv", "kps6e09-xgb-sample", 0.94638, "4bfa010a8cb0"),
    ("najiama LGBM v6", "submission_LIGHTGBM.csv", "pure-lgbm", 0.94638, "8d086828dd87"),
    ("heuljax logistic regression", "submission.csv", "logistic-regression-sample", 0.94637, "5467e97d8f20"),
    ("mizushima LGBM", "submission.csv", "competition-only", 0.94636, "3688a53cc636"),
    ("aryankaisth HistGBM", "test_proba.csv", "signal-that-matters", 0.94636, "38c0e0664fa9"),
    ("yekenot RealMLP", "submission.csv", "realmlp-pytabkit", 0.94621, "13ab8a40a522"),
    ("jazivxt R model", "submission.csv", "zoom-zoom-r-edition", 0.94621, "a2ccafc048a2"),
    ("evgendvorkin XGB", "submission.csv", "single-xgb-cv-0-94607", 0.94620, "d51dd7052f3d"),
    ("tamerlan fingerprint stack", "submission.csv", "946072", 0.94612, "48b807e603f4"),
    ("kirill0212 LGBM", "submission.csv", "s6e9-lightgbm", 0.94607, "5d85442683e0"),
    ("crystalbaby teacher-student", "submission.csv", "pseudo-labels", 0.94656, "23052891f418"),
]
PROBED = {"23052891f418": "copy of megayak's public-split probe"}


def probit(values):
    return norm.ppf(rankdata(values) / (len(values) + 1))


TOWER, lines = [], []
for label, pattern, hint, public, pin in GRID:
    try:
        path = find(pattern, hint)
    except FileNotFoundError:
        lines.append(f"{label:31s} {'-' * 12}  missing input")
        continue
    digest = sha12(path)
    if digest in PROBED:
        verdict = "dropped: " + PROBED[digest]
    elif digest != pin:
        verdict = f"dropped: bytes changed since the score (expected {pin})"
    else:
        frame = pd.read_csv(path)
        column = [c for c in frame.columns if c != ID][0]
        values = frame.set_index(ID)[column].reindex(test_ids).to_numpy(np.float64)
        if not np.isfinite(values).all():
            verdict = "dropped: ids do not match the test set"
        else:
            TOWER.append((label, probit(values), public))
            verdict = "on the grid"
    lines.append(f"{label:31s} {digest}  {public:.5f}  {verdict}")
box("the grid · sha256 (first 12) · public score · verdict", lines, width=118)
pit(2, f"{len(TOWER)} files on the grid")

## Lap 3 · The pit-stop model

Rank every file and map the ranks through the inverse normal CDF, so every file lives on the same scale and no AUC
changes. Under a binormal model a file with AUC $a$ has class separation $\delta = z/\sqrt{1+\pi(1-\pi)z^2}$,
$z = \sqrt2\,\Phi^{-1}(a)$, and any weighted mix $v$ of the files has

$$\text{AUC}(v) = \Phi\!\left(\frac{v^\top\delta}{\sqrt{2\,v^\top\Sigma_w v}}\right),\qquad
\Sigma_w = \Sigma - \pi(1-\pi)\,\delta\delta^\top .$$

The catch is $\Sigma$. The plain correlation between two files is dominated by thousands of easy negatives that
every model ranks the same way, so it wildly overstates how differently they rank the rows that matter. Versions 3–7
of this notebook measured the fix on 66 labeled pairs of out-of-fold predictions: take the correlation **inside the
top half of the ranking** and shrink its distance from 1 by a constant,

$$1-\rho_\text{eff} = k\,\bigl(1-\rho_\text{top 50\%}\bigr), \qquad k = 0.285 \;(\pm 4\%).$$

| version | predicted before or without seeing it | measured |
|---|---:|---:|
| v1 | 0.946556 | 0.94655 |
| v2 | 0.946573 | 0.94657 |
| v3 | 0.946576 | 0.94657 |
| v4 / v5 | 0.946589 | 0.94659 |
| v6 / v7 | 0.946602 | 0.94661 |
| v8 | 0.946634 | 0.94663 |
| v9 | 0.946636 | 0.94663 |
| v10 | 0.946636 | 0.94663 |

🎓 **Pit-stop lesson: when a closed form fails, find the missing constant on data where you know the truth.**

In [ ]:
def top_half_corr(g1, g2):
    top = rankdata(g1 + g2) > len(g1) / 2
    return np.corrcoef(g1[top], g2[top])[0, 1]


def separation(auc, pi=PRIOR_POS_RATE):
    z = np.sqrt(2) * norm.ppf(auc)
    return z / np.sqrt(1 + pi * (1 - pi) * z * z)


m = len(TOWER)
R = np.eye(m)
for i in range(m):
    for j in range(i + 1, m):
        R[i, j] = R[j, i] = 1.0 - K_TOP * (1.0 - top_half_corr(TOWER[i][1], TOWER[j][1]))
d_sep = np.array([separation(t[2]) for t in TOWER])
S_w = R - PRIOR_POS_RATE * (1 - PRIOR_POS_RATE) * np.outer(d_sep, d_sep)
anchor_ix = [t[0] for t in TOWER].index("jazivxt anchor")
print("how differently each file ranks the contested rows, versus the anchor (1 - effective ρ, ×10⁴):")
for (label, vec, public), r in sorted(zip(TOWER, R[anchor_ix]), key=lambda z: z[1]):
    dist = (1 - r) * 1e4
    print(f"  {label:31s} {'#' * int(round(dist * 4)):32s} {dist:5.2f}   public {public:.5f}")
pit(3, "pit-stop model loaded")

## Lap 4 · The timing tower

Maximize the predicted public AUC over weights $v \ge 0$ that sum to 1. Two guard rails keep the mix from leaning
on one noisy measurement: a file scored at 0.94650 or better may carry up to 90%, any other file at most 30%.

The model wants what every good blend wants: strong files that **disagree** on the contested rows. The files on top
of the Code tab agree with each other almost perfectly (their bars above are tiny). The logistic regression from
heuljax (Paul Bryan Elefante) and BlamerX's window-encoding XGBoost rank those rows most differently while still
scoring above 0.9463, so they earn their seats.

🎓 **Pit-stop lesson: at the top of a leaderboard, diversity is the scarce resource, not accuracy.**

In [ ]:
bounds = [(0.0, 0.90) if t[2] >= 0.94650 else (0.0, 0.30) for t in TOWER]
rng = np.random.default_rng(SEED)
best = None
for _ in range(40):
    res = minimize(lambda v: -1e3 * (v @ d_sep) / np.sqrt(2.0 * v @ S_w @ v), rng.dirichlet(np.ones(m)),
                   method="SLSQP", bounds=bounds, constraints=[{"type": "eq", "fun": lambda v: v.sum() - 1.0}],
                   options={"ftol": 1e-12, "maxiter": 600})
    if res.success and (best is None or res.fun < best.fun):
        best = res
W = np.clip(best.x, 0.0, None)
W[W < 0.005] = 0.0
W /= W.sum()


def tower_auc(v):
    return float(norm.cdf(v @ d_sep / np.sqrt(2.0 * v @ S_w @ v)))


PRED = tower_auc(W)
print(f"  P  {'file':31s} public   weight")
for pos, i in enumerate(np.argsort([-t[2] for t in TOWER]), 1):
    label, vec, public = TOWER[i]
    bar = "=" * int(round(W[i] * 40)) + ("o=o>" if W[i] > 0 else "")
    print(f" {pos:2d}  {label:31s} {public:.5f}  {W[i]:5.1%} {bar}")
print(f"\npredicted public AUC of the mix: {PRED:.6f}   (best single file: {max(t[2] for t in TOWER):.5f})")
pit(4, "timing tower solved")

## Lap 5 · Round+Robin, and rules of the road

**What nina2025's Round+Robin does.** It takes this notebook's version-8 file (99%) and jazivxt's 0.94660 blend (1%),
mixes them with an *h-blend*, adds 0.25% of a small three-model blend, and finally rounds every prediction to four
decimals. The h-blend is a clever idea: for each row it sorts the files' predictions and gives the larger one a slightly
bigger weight (+0.7 percentage points here), a soft "trust the more confident file" rule. The rounding turns 286,571
distinct scores into about 10,000, so rows tie in groups of about 29.

**How we use it.** Its file scored 0.94663, the same as versions 8 and 9. It is a near-copy of our own mix (rank
correlation 0.999997), measured once more on the public rows, so it joins the grid as a scored file and the tower
decides its weight. We don't copy the rounding. AUC gives a tied positive/negative pair half a point. Rounding only
ever *removes* ordering information, so on average it can only cost AUC, even if a single public split happens to
reward it. The cell below counts what rounding would do to our own final file.

🎓 **Pit-stop lesson: a trick that ties the leader is not a trick that beats it.** Test every transformation against
the one number that measures it, and keep only what moves it.

The generator has four hard cells that are perfectly pure in 668k training rows. Every blend leaks a few of their
rows out of the extreme blocks, so we push each cell to the very top or bottom and keep the order inside it. In
versions 2–7 this was validated fold by fold on out-of-fold predictions: **+0.0000057 AUC, positive on 5 of 5 folds**.

| rule | direction |
|---|---|
| income ≥ 170,537 | always buys → top |
| income 31,004 – 41,970 | never buys → bottom |
| daily commute ≥ 83 km | never buys → bottom |
| income exactly 30,000, no subsidy, concern 1 or Medium/High anxiety | never buys → bottom |

In [ ]:
def rule_masks(frame):
    income = frame["Annual_Income_USD"].to_numpy(np.float64)
    commute = frame["Daily_Commute_km"].to_numpy(np.float64)
    no_subsidy = frame["Subsidy_Available"].astype(str).to_numpy() == "No"
    concern_1 = frame["Environmental_Concern_Level"].to_numpy() == 1
    anxious = np.isin(frame["Range_Anxiety_Level"].astype(str).to_numpy(), ["Medium", "High"])
    return [
        ("income >= 170,537 -> always buys", income >= 170537, +1),
        ("income 31,004-41,970 -> never buys", (income >= 31004) & (income <= 41970), -1),
        ("commute >= 83 km -> never buys", commute >= 83, -1),
        ("30k, no subsidy, concern 1 or anxious", (income == 30000) & no_subsidy & (concern_1 | anxious), -1),
    ]


def apply_rules(score, frame):
    r = rankdata(score) / len(score)
    for _, mask, sign in rule_masks(frame):
        r[mask] += 2.0 * sign
    return rankdata(r) / len(r)


merged = sum(w * t[1] for w, t in zip(W, TOWER) if w > 0)
before = rankdata(merged) / len(merged)
final = apply_rules(merged, test)
rounded = np.round(final, 4)
print(f"Round+Robin check: rounding our final file to 4 decimals would leave {len(np.unique(rounded)):,} distinct scores "
      f"out of {len(final):,} (ties of ~{len(final) / len(np.unique(rounded)):.0f} rows). We keep full precision.")
for label, mask, sign in rule_masks(test):
    print(f"  {label:40s} test rows {mask.sum():6,d} · blend rank {before[mask].min():.4f}–{before[mask].max():.4f}"
          f" → {final[mask].min():.4f}–{final[mask].max():.4f}")
pit(5, "rules applied")

## Lap 6 · The honest hedge: an out-of-fold stack

Everything so far chases the **public** 20%. A good final pair needs a second file that never looked at the public
board. [goodpjw2008](https://www.kaggle.com/code/goodpjw2008/s6e9-lr-margin-gbdt-oof-stack-lb-0-94675) showed the way:
take the ridge logistic regression above, **boost GBDTs from its logit** (`init_score` / `base_margin`), and stack them
with public out-of-fold predictions using weights fitted *inside* a nested cross-validation.

This lap does that with fourteen members whose out-of-fold predictions are public: goodpjw2008's GLM and two residual
GBDTs, the ridge logistic regression, heuljax's and BlamerX's XGBoosts, megayak's views, denpugovkin's v19, najiama's
blend and three components of my Fusion IV. A Caruana hill climb picks the weights on 4/5 of the rows and is scored on
the held-out fifth, five times. The generator rules are applied last, and the result is written to
`submission_oof_stack.csv`.

🎓 **Pit-stop lesson: a model that sees the rows differently is worth more than a model that sees them better.** The
ridge LR is weaker than several GBDTs out of fold, yet it earns one of the largest weights in both the public tower and
the honest stack.

In [ ]:
from sklearn.model_selection import StratifiedKFold

train = pd.read_csv(find("train.csv", "playground-series-s6e9"), usecols=[ID, TARGET] + RULE_COLS)
y = train[TARGET].map({"No": 0, "Yes": 1}).to_numpy(np.int8)
yb = y.astype(bool)
train_ids = train[ID].to_numpy()


def col_of(frame, ids, column):
    values = frame.set_index(ID)[column].reindex(ids).to_numpy(np.float64)
    assert np.isfinite(values).all(), column
    return values


def parquet_in(hint, kind):
    hits = sorted(p for p in ROOT.rglob("*.parquet") if p.is_file() and hint in str(p) and kind in str(p).upper())
    return pd.read_parquet(hits[0])


H = {}
fx_oof, fx_test = pd.read_csv(find("oof_components.csv", "fusion-iv")), pd.read_csv(find("test_components.csv", "fusion-iv"))
for c in ("fusion", "C1_chinzo_lgbm", "D_megayak_liftwin"):
    H["Fusion IV " + c] = (col_of(fx_oof, train_ids, c), col_of(fx_test, test_ids, c))
six_oof, six_test = pd.read_csv(find("oof_six_views.csv")), pd.read_csv(find("test_six_views.csv"))
mlp_oof, mlp_test = pd.read_csv(find("oof_realmlp_g.csv")), pd.read_csv(find("test_realmlp_g.csv"))
H["megayak ensemble"] = (col_of(six_oof, train_ids, "ensemble"), col_of(six_test, test_ids, "ensemble"))
H["megayak view D"] = (col_of(six_oof, train_ids, "D_no_exact_key_ladder_windows"),
                       col_of(six_test, test_ids, "D_no_exact_key_ladder_windows"))
H["megayak RealMLP 3-seed"] = (col_of(mlp_oof, train_ids, "G_realmlp_3seed"), col_of(mlp_test, test_ids, "G_realmlp_3seed"))
res_dir = find("v19_oof.csv", "residual-stack").parent
H["denpugovkin v19"] = (col_of(pd.read_csv(res_dir / "v19_oof.csv"), train_ids, "pred"),
                        col_of(pd.read_csv(res_dir / "v19_test.csv"), test_ids, "pred"))
H["najiama blend"] = (col_of(pd.read_csv(find("01_blend_oof.csv", "s6e9-oof")), train_ids, TARGET),
                      col_of(pd.read_csv(find("01_submission.csv", "s6e9-oof")), test_ids, TARGET))
H["BlamerX XGB"] = (col_of(pd.read_csv(find("oof.csv", "window-encodings")), train_ids, "pred"),
                    col_of(pd.read_csv(find("submission.csv", "window-encodings")), test_ids, TARGET))
H["heuljax XGB"] = (col_of(parquet_in("kps6e09-xgb-sample", "OOF"), train_ids, "oof_pred"),
                    col_of(parquet_in("kps6e09-xgb-sample", "TEST"), test_ids, "test_pred"))
H["heuljax ridge LR"] = (col_of(parquet_in("generator-aware-ridge", "OOF"), train_ids, "oof_pred"),
                         col_of(parquet_in("generator-aware-ridge", "TEST"), test_ids, "test_pred"))
gp_oof, gp_test = pd.read_csv(find("oof_mine.csv", "lr-margin")), pd.read_csv(find("test_mine.csv", "lr-margin"))
for c in ("glm", "residual_lgbm", "residual_xgb"):
    H["goodpjw2008 " + c] = (col_of(gp_oof, train_ids, c), col_of(gp_test, test_ids, c))

HN = list(H)
HO = np.column_stack([probit(H[k][0]) for k in HN])
HT = np.column_stack([probit(H[k][1]) for k in HN])


def fast_auc(labels, score):
    order = np.argsort(score)
    ranked = labels[order]
    n_pos = ranked.sum()
    n_neg = len(ranked) - n_pos
    return (np.flatnonzero(ranked).sum() + n_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def hill_climb(P, labels, max_steps=60, tol=1e-7):
    blend, counts, best = np.zeros(len(P)), np.zeros(P.shape[1]), -1.0
    for _ in range(max_steps):
        scores = [fast_auc(labels, blend + P[:, j]) for j in range(P.shape[1])]
        j = int(np.argmax(scores))
        if scores[j] <= best + tol:
            break
        best, counts[j] = scores[j], counts[j] + 1
        blend += P[:, j]
    return counts / counts.sum()


single = {k: fast_auc(yb, HO[:, i]) for i, k in enumerate(HN)}
pole = max(single, key=single.get)
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(HO, y))
stack_oof = np.zeros(len(y))
for k, (tr, va) in enumerate(folds):
    stack_oof[va] = HO[va] @ hill_climb(HO[tr], yb[tr])
    print(f"  fold {k}: stack {fast_auc(yb[va], stack_oof[va]):.6f} vs {pole} {fast_auc(yb[va], HO[va, HN.index(pole)]):.6f}")
W_H = hill_climb(HO, yb)
stack_ruled = apply_rules(stack_oof, train)
print(f"\nbest single member {pole} {single[pole]:.6f} · nested stack {fast_auc(yb, stack_oof):.6f}"
      f" · with rules {fast_auc(yb, stack_ruled):.6f}")
for k, w in sorted(zip(HN, W_H), key=lambda z: -z[1]):
    if w > 0:
        print(f"  {k:30s} {'#' * int(round(w * 40)):40s} {w:.3f}")
hedge = apply_rules(HT @ W_H, test)
pd.DataFrame({ID: test_ids, TARGET: hedge}).to_csv(OUT / "submission_oof_stack.csv", index=False)
pit(6, "honest hedge written")

In [ ]:
submission = pd.DataFrame({ID: test_ids, TARGET: final})
assert submission[ID].is_unique and len(submission) == len(sample) and submission[TARGET].between(0, 1).all()
submission.to_csv(OUT / "submission.csv", index=False)
pit(7, "checkered flag")

mix = [(t[0], w, t[2]) for t, w in sorted(zip(TOWER, W), key=lambda z: -z[1]) if w > 0]
box("results · version 11", [
    f"predicted public AUC ........ {PRED:.6f}",
    f"files on the grid ........... {len(TOWER)}",
    f"runtime on CPU .............. {time.time() - T0:.0f}s",
    f"submission.csv sha256 ....... {sha12(OUT / 'submission.csv')}",
    "",
    "final mix                                   weight    public AUC",
] + [f"  {label:40s} {w:6.1%}    {public:.5f}" for label, w, public in mix] + [
    "",
    "rules of the road applied after the mix",
    "",
    f"honest hedge (Lap 6) · nested OOF {fast_auc(yb, stack_oof):.6f} · with rules {fast_auc(yb, stack_ruled):.6f}",
    "submission.csv = public challenger · submission_oof_stack.csv = labels only",
])
top3 = (mix + [("-", 0.0, 0.0)] * 3)[:3]
podium(*[(label, f"{w:5.1%} of the mix · public {public:.5f}" if w else "") for label, w, public in top3])
print(TROPHY)

## 🏆 Takeaways

1. **Blend on one scale.** Probit ranks put every file on the same track without changing any AUC.
2. **Calibrate the closed form.** The binormal Fisher model is only as good as its correlation. The top-half
   correlation with $k = 0.285$ predicted all seven earlier versions of this notebook within 0.00001.
3. **Diversity is the scarce resource.** The best files at the top of the Code tab are near-copies of each other. A
   0.94637 logistic regression that ranks the contested rows differently is worth more to the mix than another 0.9466.
4. **Pin versions and hash bytes.** A score belongs to one version; an attached notebook mounts the latest one.
5. **Rules of the road** are free AUC once they are validated on labels. Rounding is not: ties only remove ordering.
6. **Know which leaderboard you are optimizing.** `submission.csv` chases the public 20%. `submission_oof_stack.csv`
   never saw a public score. Keep one of each as your final pair.
7. **Diversity beats accuracy at the top.** A 0.94656 logistic regression outweighs every 0.9466 blend in the mix.

### Credits
Every file on the grid belongs to its author: [@jazivxt](https://www.kaggle.com/jazivxt) (anchor, blend, R model),
[@nina2025](https://www.kaggle.com/nina2025), [@tamerlanomralinov](https://www.kaggle.com/tamerlanomralinov),
[@shawncsx](https://www.kaggle.com/shawncsx), [@blamerx](https://www.kaggle.com/blamerx),
[@denpugovkin](https://www.kaggle.com/denpugovkin), [@mikhailnaumov](https://www.kaggle.com/mikhailnaumov),
[@heuljax](https://www.kaggle.com/heuljax) (Paul Bryan Elefante, XGBoost, logistic and generator-aware ridge models),
[@goodpjw2008](https://www.kaggle.com/code/goodpjw2008/s6e9-lr-margin-gbdt-oof-stack-lb-0-94675) (GLM and residual GBDTs,
the honest-stack recipe of Lap 6), [@shawncsx](https://www.kaggle.com/shawncsx), [@kozykappa](https://www.kaggle.com/kozykappa), [@nina2025](https://www.kaggle.com/code/nina2025/fork-of-ps-s6e9-round-robin) (Round+Robin, h-blend), [@najiama](https://www.kaggle.com/najiama),
[@mizushimatoshihiko](https://www.kaggle.com/mizushimatoshihiko), [@aryankaisth](https://www.kaggle.com/aryankaisth),
[@yekenot](https://www.kaggle.com/yekenot), [@evgendvorkin](https://www.kaggle.com/evgendvorkin),
[@kirill0212](https://www.kaggle.com/kirill0212), [@crystalbaby](https://www.kaggle.com/crystalbaby).
Fisher-from-scores idea: [@thisray](https://www.kaggle.com/thisray). Generator rules: [@megayak](https://www.kaggle.com/megayak)
and [@defiaudit](https://www.kaggle.com/defiaudit). The generator's recipe: [@cdeotte](https://www.kaggle.com/cdeotte).